- run ADLS access setup

In [0]:
client_id = dbutils.secrets.get(scope="financial-scope", key="adls-client-id2")
tenant_id = dbutils.secrets.get(scope="financial-scope", key="adls-tenant-id")
client_secret = dbutils.secrets.get(scope="financial-scope", key="adls-client-secret")

spark.conf.set(
    "fs.azure.account.auth.type.financialdatalakestorage.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.financialdatalakestorage.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.financialdatalakestorage.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.financialdatalakestorage.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.financialdatalakestorage.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
import requests
import json
from datetime import datetime, date

# getting the data from CoinGecko API 
url = "https://api.coingecko.com/api/v3/coins/markets"

params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1
}

response = requests.get(url, params=params)

if response.status_code != 200:
    raise Exception(f"API failed with status {response.status_code}")

data = response.json()

# Adding ingetion time to each record 
ingestion_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

for record in data:
    record["ingestion_time"] = ingestion_time

# Writing the data into ADLS bronze layer
today = date.today().strftime("%Y-%m-%d")
base_path = f"abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/bronze/crypto/ingestion_date={today}"
file_path = f"{base_path}/raw_data.json"
dbutils.fs.put(file_path, json.dumps(data), overwrite=True)

print(f"Data written to: {file_path}")

/home/spark-20e320bd-f9e3-4821-8ac9-a3/.ipykernel/3826/command-6757412003708830-3275295230:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingestion_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


Wrote 8847 bytes.
Data written to: abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/bronze/crypto/ingestion_date=2026-05-02/raw_data.json
